# Блок 1 — Классификация: Decision Tree и Random Forest


In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
df = pd.read_csv("train.csv")

print("Размер датасета:", df.shape)
print("\nПервые 5 строк:")
print(df.head())

print("\nИнформация о датасете:")
print(df.info())

print("\nПропуски:")
print(df.isnull().sum().sort_values(ascending=False).head(15))

Размер датасета: (1460, 81)

Первые 5 строк:
   Id  MSSubClass MSZoning  LotFrontage  LotArea Street Alley LotShape  \
0   1          60       RL         65.0     8450   Pave   NaN      Reg   
1   2          20       RL         80.0     9600   Pave   NaN      Reg   
2   3          60       RL         68.0    11250   Pave   NaN      IR1   
3   4          70       RL         60.0     9550   Pave   NaN      IR1   
4   5          60       RL         84.0    14260   Pave   NaN      IR1   

  LandContour Utilities  ... PoolArea PoolQC Fence MiscFeature MiscVal MoSold  \
0         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      2   
1         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      5   
2         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      9   
3         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      2   
4         Lvl    AllPub  ...        0    NaN   NaN         NaN       0     12   

  YrSold  SaleType  Sal

In [ ]:
data = df[['GrLivArea', 'OverallQual', 'GarageCars', 'TotalBsmtSF', 'SalePrice']].copy()
data = data.dropna()

print(data.head())
print("\nРазмер после удаления пропусков:", data.shape)

   GrLivArea  OverallQual  GarageCars  TotalBsmtSF  SalePrice
0       1710            7           2          856     208500
1       1262            6           2         1262     181500
2       1786            7           2          920     223500
3       1717            7           3          756     140000
4       2198            8           3         1145     250000

Размер после удаления пропусков: (1460, 5)


# Задача 3. Классификация: дерево решений и Random Forest

Цель: предсказать, является ли цена дома выше медианной (`SalePrice > median`).

In [ ]:
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

# Для классификации делаем бинарную цель: выше медианы/ниже или равно медиане
price_threshold = data['SalePrice'].median()
data_clf = data.copy()
data_clf['HighPrice'] = (data_clf['SalePrice'] > price_threshold).astype(int)

X_clf = data_clf[['GrLivArea', 'OverallQual', 'GarageCars', 'TotalBsmtSF']]
y_clf = data_clf['HighPrice']

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

print(f"Порог медианы SalePrice: {price_threshold:.2f}")
print("Класс 1 = цена выше медианы")

In [ ]:
clf_tree = DecisionTreeClassifier(max_depth=5, random_state=42)
clf_tree.fit(Xc_train, yc_train)
yc_pred_tree = clf_tree.predict(Xc_test)

clf_rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)
clf_rf.fit(Xc_train, yc_train)
yc_pred_rf = clf_rf.predict(Xc_test)

clf_metrics = pd.DataFrame({
    'Модель': ['Decision Tree Classifier', 'Random Forest Classifier'],
    'Accuracy': [accuracy_score(yc_test, yc_pred_tree), accuracy_score(yc_test, yc_pred_rf)],
    'F1': [f1_score(yc_test, yc_pred_tree), f1_score(yc_test, yc_pred_rf)]
})

print(clf_metrics)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(yc_test, yc_pred_tree),
    display_labels=['<= median', '> median']
).plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Decision Tree: Confusion Matrix')

ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(yc_test, yc_pred_rf),
    display_labels=['<= median', '> median']
).plot(ax=axes[1], cmap='Greens', colorbar=False)
axes[1].set_title('Random Forest: Confusion Matrix')

plt.tight_layout()
plt.show()